In [ ]:
from fastapi.responses import StreamingResponse
import json

@app.post("/chat-stream")
async def chat_stream(req: Request):
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)

    async def generator():
        async with semaphore:
            async with httpx.AsyncClient(timeout=None) as client:

                payload = {
                    "model": "PatronusAI/lynx3_4b_full_finetune_v0.995-fp8-dynamic",
                    "messages": [{"role": "user", "content": req.prompt}],
                    "temperature": 0.2,
                    "stream": True
                }

                async with client.stream("POST", VLLM_URL, json=payload) as resp:
                    async for line in resp.aiter_lines():
                        if line:
                            yield line + "\n"

    return StreamingResponse(generator(), media_type="text/event-stream")